# UK Online Retail — Business Analysis
**Alwyn Fernandus** | Portfolio Project

---

**Dataset:** UCI Online Retail — 541,909 transactions, Dec 2010 – Dec 2011  
**Source:** [archive.ics.uci.edu/dataset/352/online+retail](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Tools:** Python · pandas · matplotlib · seaborn · scikit-learn

---

## Background

This is a UK-based online retailer that sells all-occasion gifts — mostly to wholesale buyers, but also to individual customers across 37 countries. The dataset covers about 13 months of invoiced transactions.

The company had revenue data, but no real picture of customer behaviour underneath it. The goal here was to build that picture: who's buying, how often, which products are pulling their weight, and where the marketing effort is and isn't aligned with actual buyer activity.

**Stakeholders this analysis is relevant to:** Marketing, E-Commerce, Finance, Customer Success


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Consistent colour palette across all charts
PALETTE = {
    'primary'  : '#20808D',
    'secondary': '#A84B2F',
    'dark'     : '#1B474D',
    'light'    : '#BCE2E7',
    'accent'   : '#FFC553',
    'muted'    : '#7A7974',
    'bg'       : '#F7F6F2',
    'text'     : '#28251D',
}

plt.rcParams.update({
    'figure.facecolor' : PALETTE['bg'],
    'axes.facecolor'   : '#FFFFFF',
    'axes.edgecolor'   : '#D4D1CA',
    'axes.titlesize'   : 14,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'grid.color'       : '#E8E7E3',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'figure.dpi'       : 120,
})

print("Ready")

## 1. Load Data

In [ ]:
df_raw = pd.read_excel('../data/Online Retail.xlsx', engine='openpyxl')

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_raw.head(3)

## 2. Data Cleaning

A few things needed sorting before the analysis could start. The notes below explain what was removed and why.


In [ ]:
# First, check what's missing
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
summary = pd.DataFrame({'Count': missing, 'Pct': missing_pct})
print(summary[summary['Count'] > 0].to_string())

In [ ]:
df = df_raw.copy()

# 135K rows have no CustomerID — these are guest/anonymous sessions.
# We can't track behaviour without an ID, so they're out.
df = df.dropna(subset=['CustomerID'])
print(f"After dropping null CustomerID : {len(df):,}")

# Invoices starting with 'C' are cancellations, not real orders
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations   : {len(df):,}")

# A small number of rows have negative quantities or zero prices — clearly erroneous
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"After removing bad qty/price   : {len(df):,}")

# A handful of internal stock codes that aren't real products (postage, bank charges, etc.)
test_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df = df[~df['StockCode'].isin(test_codes)]
print(f"After removing internal codes  : {len(df):,}")

# Type cleanup and derived columns
df = df.copy()
df['CustomerID']  = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Revenue']   = df['Quantity'] * df['UnitPrice']
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['Month']     = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour']      = df['InvoiceDate'].dt.hour

print(f"\nClean dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

In [ ]:
print(f"Date range  : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
print(f"Customers   : {df['CustomerID'].nunique():,}")
print(f"Products    : {df['StockCode'].nunique():,}")
print(f"Countries   : {df['Country'].nunique()}")
print(f"Revenue     : £{df['Revenue'].sum():,.0f}")
print(f"Orders      : {df['InvoiceNo'].nunique():,}")
df.describe().T

## 3. KPI Baseline

Before getting into the charts, it helps to have a clear baseline of the top-line numbers. These are the metrics that matter most from a business health perspective.

| KPI | How It's Calculated | Why It Matters |
|-----|---------------------|----------------|
| **Total Revenue** | Quantity × Unit Price, summed | The obvious one — overall business size |
| **Average Order Value** | Revenue ÷ Orders | Signals pricing power and basket size |
| **Repeat Customer Rate** | Customers with 2+ orders ÷ total | Shows whether the product keeps people coming back |
| **Orders per Customer** | Total orders ÷ unique customers | Depth of engagement beyond first purchase |
| **RFM Segments** | Scored on Recency, Frequency, Monetary | Tells you where to focus retention effort |
| **Cohort Retention** | % of each cohort returning month-over-month | Structural loyalty, not just one-off numbers |
| **Revenue Concentration** | Champions' share of total revenue | How fragile the revenue base really is |
| **International Share** | Non-UK revenue ÷ total | Signals global growth potential |


In [ ]:
snapshot_date   = df['InvoiceDate'].max() + pd.Timedelta(days=1)
total_revenue   = df['Revenue'].sum()
total_orders    = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
aov             = total_revenue / total_orders
orders_per_cust = total_orders / total_customers
cust_freq       = df.groupby('CustomerID')['InvoiceNo'].nunique()
repeat_pct      = (cust_freq > 1).sum() / total_customers * 100
uk_pct          = df[df['Country'] == 'United Kingdom']['Revenue'].sum() / total_revenue * 100

pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Orders', 'Customers',
               'Avg. Order Value', 'Orders per Customer',
               'Repeat Customer Rate', 'UK Revenue Share'],
    'Value': [f'£{total_revenue:,.0f}', f'{total_orders:,}', f'{total_customers:,}',
              f'£{aov:.2f}', f'{orders_per_cust:.1f}',
              f'{repeat_pct:.1f}%', f'{uk_pct:.1f}%']
}).style.set_caption("Business KPI Baseline")

## 4. Exploratory Analysis

### 4.1 Revenue Over Time

First thing I wanted to see was the monthly shape of the business — whether growth was steady, seasonal, or driven by a couple of big months.


In [ ]:
monthly = df.groupby('YearMonth').agg(
    Revenue  =('Revenue', 'sum'),
    Orders   =('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5), facecolor=PALETTE['bg'])
ax2 = ax1.twinx()

ax1.bar(monthly['YearMonth_str'], monthly['Revenue'] / 1000,
        color=PALETTE['primary'], alpha=0.75, width=0.6, label='Revenue (£K)')
ax2.plot(monthly['YearMonth_str'], monthly['Orders'],
         color=PALETTE['secondary'], marker='o', linewidth=2.2,
         markersize=5, label='Orders', zorder=5)

peak_idx = monthly['Revenue'].idxmax()
ax1.annotate(f"Peak\n£{monthly['Revenue'][peak_idx]/1000:.0f}K",
             xy=(peak_idx, monthly['Revenue'][peak_idx]/1000),
             xytext=(peak_idx - 1.5, monthly['Revenue'][peak_idx]/1000 * 0.85),
             arrowprops=dict(arrowstyle='->', color=PALETTE['dark'], lw=1.5),
             fontsize=8.5, color=PALETTE['dark'], fontweight='bold')

ax1.set_ylabel('Revenue (£K)', color=PALETTE['primary'], labelpad=8)
ax2.set_ylabel('Orders', color=PALETTE['secondary'], labelpad=8)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}K'))
ax1.set_title('Monthly Revenue & Order Volume', fontsize=14, fontweight='bold', pad=12)
ax1.set_facecolor('#FFFFFF')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Peak month : {monthly.loc[peak_idx, 'YearMonth_str']} — £{monthly['Revenue'][peak_idx]:,.0f}")
print(f"Nov share  : {monthly[monthly['YearMonth_str']=='2011-11']['Revenue'].values[0]/total_revenue*100:.1f}% of annual revenue")

### 4.2 Product Revenue

Worth checking whether the revenue is spread across the catalogue or concentrated in a handful of products.


In [ ]:
product_rev = (df.groupby('Description')['Revenue']
               .sum().sort_values(ascending=False)
               .head(10).reset_index())
product_rev['Description'] = product_rev['Description'].str.title().str[:40]
product_rev['Revenue_K'] = product_rev['Revenue'] / 1000

fig, ax = plt.subplots(figsize=(11, 6), facecolor=PALETTE['bg'])
colors = [PALETTE['primary'] if i == 0 else PALETTE['light'] for i in range(len(product_rev))]
bars = ax.barh(product_rev['Description'][::-1], product_rev['Revenue_K'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, product_rev['Revenue_K'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'£{val:.1f}K', va='center', ha='left', fontsize=8.5,
            color=PALETTE['text'], fontweight='bold')

ax.set_xlabel('Total Revenue (£K)')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, product_rev['Revenue_K'].max() * 1.22)
plt.tight_layout()
plt.show()

### 4.3 Geographic Breakdown

The UK is the home market, but this business ships internationally. I wanted to see which countries were generating meaningful revenue — especially without any obvious marketing investment.


In [ ]:
country_rev = (df[df['Country'] != 'United Kingdom']
               .groupby('Country')['Revenue']
               .sum().sort_values(ascending=False)
               .head(10).reset_index())
country_rev['Revenue_K'] = country_rev['Revenue'] / 1000

fig, ax = plt.subplots(figsize=(11, 6), facecolor=PALETTE['bg'])
bar_colors = [PALETTE['primary']] + [PALETTE['secondary']] * 2 + [PALETTE['light']] * 7
ax.bar(country_rev['Country'], country_rev['Revenue_K'],
       color=bar_colors[:len(country_rev)], edgecolor='white')

for i, (_, row) in enumerate(country_rev.iterrows()):
    ax.text(i, row['Revenue_K'] + 0.5, f'£{row["Revenue_K"]:.1f}K',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_ylabel('Revenue (£K)')
ax.set_title('International Revenue — Top 10 Markets (excl. UK)', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

intl_share = (1 - uk_pct/100) * 100
print(f"International share: {intl_share:.1f}% of total revenue")
print(f"Top market outside UK: {country_rev.iloc[0]['Country']} (£{country_rev.iloc[0]['Revenue']:,.0f})")

### 4.4 When Are Customers Actually Ordering?

This one matters for marketing timing. I broke down orders by day of week and hour of day to see if there's a pattern.


In [ ]:
dow_order   = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_orders  = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(dow_order).reset_index()
hour_orders = df.groupby('Hour')['InvoiceNo'].nunique().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE['bg'])

ax1.bar(day_orders['DayOfWeek'], day_orders['InvoiceNo'],
        color=[PALETTE['primary'] if d not in ['Saturday', 'Sunday'] else PALETTE['muted']
               for d in dow_order], edgecolor='white')
ax1.set_title('Orders by Day of Week', fontsize=13, fontweight='bold', pad=10)
ax1.tick_params(axis='x', rotation=30)
ax1.set_facecolor('#FFFFFF')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(hour_orders['Hour'], hour_orders['InvoiceNo'],
         color=PALETTE['secondary'], linewidth=2.5, marker='o', markersize=5)
ax2.fill_between(hour_orders['Hour'], hour_orders['InvoiceNo'], alpha=0.15, color=PALETTE['secondary'])
peak_hour = hour_orders.loc[hour_orders['InvoiceNo'].idxmax()]
ax2.annotate(f"Peak: {int(peak_hour['Hour'])}:00",
             xy=(peak_hour['Hour'], peak_hour['InvoiceNo']),
             xytext=(peak_hour['Hour'] + 1.5, peak_hour['InvoiceNo'] * 0.92),
             arrowprops=dict(arrowstyle='->', color=PALETTE['dark'], lw=1.3),
             fontsize=8.5, fontweight='bold')
ax2.set_title('Orders by Hour of Day', fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel('Hour (24h)')
ax2.set_facecolor('#FFFFFF')
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

weekend_orders = day_orders[day_orders['DayOfWeek'].isin(['Saturday', 'Sunday'])]['InvoiceNo'].sum()
weekday_orders = day_orders[~day_orders['DayOfWeek'].isin(['Saturday', 'Sunday'])]['InvoiceNo'].sum()
print(f"Weekend orders vs weekday: {weekend_orders:,} vs {weekday_orders:,} ({weekend_orders/(weekend_orders+weekday_orders)*100:.1f}% of total)")

## 5. Customer Segmentation — RFM

RFM (Recency, Frequency, Monetary) is a standard way to segment customers by their actual buying behaviour — when they last bought, how often they buy, and how much they spend. Each customer gets a score on each dimension, and the combination tells you who's valuable, who's slipping, and who's likely gone for good.

Scoring is done with quantile-based bins (1–4 scale), so the comparisons are relative to the full customer base, not arbitrary thresholds.


In [ ]:
rfm = df.groupby('CustomerID').agg(
    Recency  =('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary =('Revenue', 'sum')
).reset_index()

# Higher recency score = bought more recently (so we reverse the bin order for R)
rfm['R_Score'] = pd.qcut(rfm['Recency'],  4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  4, labels=[1, 2, 3, 4])

def segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    score = r + f + m
    if r == 4 and f >= 3 and m >= 3:   return 'Champions'
    elif r >= 3 and f >= 3:             return 'Loyal Customers'
    elif r >= 3 and f <= 2 and m <= 2:  return 'Promising'
    elif r <= 2 and f >= 3 and m >= 3:  return 'At Risk'
    elif r == 1 and f == 1:             return 'Lost'
    elif score >= 9:                    return 'Loyal Customers'
    elif score >= 7:                    return 'Promising'
    elif score >= 5:                    return 'Need Attention'
    else:                               return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)

seg_summary = rfm.groupby('Segment').agg(
    Customers    =('CustomerID', 'count'),
    Avg_Recency  =('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary =('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(1).sort_values('Total_Revenue', ascending=False)

seg_summary['Revenue_Share_%'] = (
    seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100
).round(1)

seg_summary.style.background_gradient(subset=['Total_Revenue'], cmap='YlGnBu').format({
    'Total_Revenue': '£{:,.0f}', 'Revenue_Share_%': '{:.1f}%'
})

In [ ]:
seg_colors = {
    'Champions'      : '#20808D',
    'Loyal Customers': '#1B474D',
    'Promising'      : '#FFC553',
    'At Risk'        : '#A84B2F',
    'Need Attention' : '#944454',
    'Lost'           : '#BAB9B4',
}
seg_counts  = rfm['Segment'].value_counts()
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=PALETTE['bg'])

seg_cnt = seg_counts.sort_values(ascending=True)
ax1.barh(seg_cnt.index, seg_cnt.values,
         color=[seg_colors.get(s, PALETTE['muted']) for s in seg_cnt.index], edgecolor='white')
for i, (s, v) in enumerate(zip(seg_cnt.index, seg_cnt.values)):
    ax1.text(v + 8, i, f'{v:,}', va='center', fontsize=9, fontweight='bold')
ax1.set_title('Customers by Segment', fontsize=13, fontweight='bold', pad=10)
ax1.set_facecolor('#FFFFFF')
ax1.grid(axis='x', linestyle='--', alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

seg_rev = seg_revenue.sort_values(ascending=True)
ax2.barh(seg_rev.index, seg_rev.values / 1000,
         color=[seg_colors.get(s, PALETTE['muted']) for s in seg_rev.index], edgecolor='white')
for i, (s, v) in enumerate(zip(seg_rev.index, seg_rev.values)):
    ax2.text(v/1000 + 1, i, f'£{v/1000:.0f}K', va='center', fontsize=9, fontweight='bold')
ax2.set_title('Revenue by Segment', fontsize=13, fontweight='bold', pad=10)
ax2.set_facecolor('#FFFFFF')
ax2.grid(axis='x', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

champ_rev = rfm[rfm['Segment'] == 'Champions']['Monetary'].sum()
print(f"Champions: {seg_counts.get('Champions', 0)} customers, £{champ_rev:,.0f} ({champ_rev/total_revenue*100:.1f}% of revenue)")

### 5.1 Recency vs. Value Scatter

This chart helps visualise the segment logic — Champions cluster top-left (recent + high value). As recency increases, value drops off. The At-Risk customers are the interesting ones: they're to the right (haven't bought recently) but some have meaningful spend behind them.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=PALETTE['bg'])

for seg, grp in rfm.groupby('Segment'):
    ax.scatter(grp['Recency'], grp['Monetary'] / 1000,
               label=seg, alpha=0.55, s=28,
               color=seg_colors.get(seg, PALETTE['muted']), edgecolors='none')

ax.set_xlabel('Days Since Last Purchase (Recency)', labelpad=8)
ax.set_ylabel('Total Spend — £K (Monetary)', labelpad=8)
ax.set_title('Recency vs. Customer Value by Segment', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, framealpha=0.9, title='Segment', title_fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

### 5.2 RFM Correlation

Just checking whether the three RFM dimensions are independent or whether some are predictable from others.


In [ ]:
corr_df = rfm[['Recency', 'Frequency', 'Monetary']].corr()

fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE['bg'])
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap=cmap,
            center=0, square=True, linewidths=2, linecolor=PALETTE['bg'],
            annot_kws={'size': 12, 'weight': 'bold'},
            ax=ax, cbar_kws={'shrink': 0.75})
ax.set_title('RFM Correlation', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
plt.tight_layout()
plt.show()

print(f"F-M correlation: {corr_df.loc['Frequency', 'Monetary']:.2f}")
print("Note: Frequency and Monetary are strongly correlated — customers who order more often also tend to spend more overall.")

## 6. Cohort Retention

RFM tells you where customers are now. Cohort analysis shows how long they've been sticking around since they first bought. I tracked each acquisition cohort (by first purchase month) and looked at what fraction returned in the following months.


In [ ]:
cohort_df = df[['CustomerID', 'YearMonth']].copy()
cohort_df['CohortMonth'] = cohort_df.groupby('CustomerID')['YearMonth'].transform('min')
cohort_df['CohortIndex'] = (
    cohort_df['YearMonth'].apply(lambda x: x.ordinal) -
    cohort_df['CohortMonth'].apply(lambda x: x.ordinal)
)

cohort_counts = (cohort_df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID']
                 .nunique().reset_index())
cohort_pivot  = cohort_counts.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size   = cohort_pivot.iloc[:, 0]
cohort_pct    = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100

cohort_plot = cohort_pct.iloc[:, :7].copy()
cohort_plot.index = cohort_plot.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 7), facecolor=PALETTE['bg'])
sns.heatmap(cohort_plot, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, linecolor=PALETTE['bg'],
            annot_kws={'size': 9}, ax=ax,
            cbar_kws={'label': 'Retention %', 'shrink': 0.8},
            vmin=0, vmax=100)
ax.set_title('Customer Retention by Cohort (First 6 Months)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Months Since First Purchase', labelpad=8)
ax.set_ylabel('Acquisition Cohort', labelpad=8)
plt.tight_layout()
plt.show()

# Month-1 retention across cohorts
m1_retention = cohort_pct.iloc[:, 1].dropna()
print(f"Average Month-1 retention: {m1_retention.mean():.1f}%")
print(f"Range: {m1_retention.min():.0f}% – {m1_retention.max():.0f}%")

## 7. Order Value Distribution

A quick look at how order values are distributed — useful context before recommending any changes to pricing or bundling.


In [ ]:
order_rev = df.groupby('InvoiceNo')['Revenue'].sum()
cap = order_rev.quantile(0.99)  # cap at 99th percentile for chart readability

fig, ax = plt.subplots(figsize=(10, 5), facecolor=PALETTE['bg'])
ax.hist(order_rev[order_rev <= cap], bins=60,
        color=PALETTE['primary'], alpha=0.8, edgecolor='white')
ax.axvline(order_rev.median(), color=PALETTE['secondary'], linestyle='--',
           linewidth=2, label=f'Median: £{order_rev.median():.0f}')
ax.axvline(order_rev.mean(), color=PALETTE['accent'], linestyle='--',
           linewidth=2, label=f'Mean: £{order_rev.mean():.0f}')
ax.set_xlabel('Order Value (£)')
ax.set_ylabel('Number of Orders')
ax.set_title('Order Value Distribution (capped at 99th percentile)', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=9.5)
plt.tight_layout()
plt.show()

print(f"Median: £{order_rev.median():.2f}")
print(f"Mean  : £{order_rev.mean():.2f}")
print(f"95th  : £{order_rev.quantile(0.95):.2f}")
print("\nThe gap between median and mean suggests a few large orders (likely wholesale) are pulling the average up.")

## 8. Insights

These are the patterns that stood out. They're not all surprising, but several of them have clear actions attached.

---

### November is carrying too much weight
Revenue peaked in November at £1.14M — about 13% of the full year in a single month. The underlying Q4 trajectory from September onwards is strong, but the business has no formal planning mechanism for this seasonality. A supply issue or delayed campaign in October would have a disproportionate impact on annual results.

---

### Champions are almost the whole business
800 customers in the Champions segment — roughly 18% of the total base — generated 55% of revenue. They buy often, they buy recently, and they spend a lot. There's no loyalty program, no early access, no personalised communication keeping them engaged. This is the single highest-risk gap in the customer strategy.

---

### One in three customers never came back
34.7% of customers made one purchase and didn't return. The cohort analysis backs this up — Month 1 retention averaged around 20%, which means 80% of any given cohort stops buying after the first month. That's a post-purchase problem. Whatever the follow-up looks like today, it's not working well enough.

---

### The At-Risk segment is worth chasing
449 customers who used to spend heavily have gone quiet (recency scores of 1–2). Their combined historical spend is £931K. These aren't random new leads — they already showed willingness to spend. A targeted re-engagement sequence would be worth the effort even at a modest success rate.

---

### Germany and Netherlands are quietly buying without help
The UK is 82.9% of revenue, but Germany (£284K) and Netherlands (£262K) were the strongest international markets — both buying regularly with no obvious targeted investment behind them. A bit of effort there seems likely to pay off.

---

### Weekend ads are probably wasted
Saturday and Sunday together accounted for well under 20% of weekly orders. Most of the volume happens on Tuesday through Thursday, between roughly 10am and 2pm. Given the B2B/wholesale profile suggested by the data, digital ads on weekends are likely reaching the wrong people at the wrong time.


## 9. Recommendations

I've grouped these by how quickly they could realistically be acted on.

### Do now (0–3 months)

| What | Who It Helps | Why |
|------|-------------|-----|
| Start a Champions loyalty program — even something simple (early access, personal thank-you discount) | Champions | Protects 55% of revenue with minimal cost |
| Build a 3-email win-back flow for At-Risk customers | At Risk | £140K recoverable at 15% conversion |
| Shift paid ad scheduling to Tue–Thu 10am–2pm | All channels | Better match with actual buyer windows |
| Add a Day 7 and Day 30 email to new customer journeys | New buyers | Targets the 35% who bought once and left |

---

### Next 3–9 months

- **RFM in the CRM:** If segment scores were updated monthly and mapped to CRM fields, the marketing team could trigger campaigns automatically when someone's score drops. No analyst needed each time.
- **Q4 prep starts in August:** November's numbers show the business performs best in Q4 but the planning doesn't match. Pre-Black Friday communications should go out by October 15, not November.
- **Product bundling:** Top products are selling well individually. Bundling them could raise the average order value without discounting.

---

### Longer term

- Run a proper Germany/Netherlands campaign. They're already converting without marketing — adding investment could meaningfully grow international revenue.
- Build a wholesale loyalty tier. The B2B buyer profile (weekday peaks, large orders) suggests these customers have different needs from retail consumers.
- A seasonal inventory model doesn't need to be complicated — even a simple linear forecast on monthly patterns would help with November planning and reduce the risk of stock-outs during the peak.


## 10. Conclusion

The core finding from this analysis is that the revenue looks healthy on the surface, but underneath there are two risks that need addressing:

1. The revenue base is heavily concentrated — Champions (800 customers) carry over half the business with no formal retention strategy around them
2. Acquisition is working, but conversion to repeat buyers is not — 35% of customers buy once and leave

The At-Risk segment adds urgency. Nearly £1M in spend from customers who've gone quiet is within reach if the business acts before that group moves fully into the Lost category.

Most of the short-term recommendations require CRM execution and ad scheduling adjustments, not new technology or significant budget. The payoff-to-effort ratio is high.

---

**Tools used:** Python · pandas · matplotlib · seaborn · scikit-learn  
**Dataset:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Alwyn Fernandus** | MBA in Business Analytics | [LinkedIn](https://www.linkedin.com/in/alwynfernandus)
